# The Forward Model

This notebook derives and demonstrates the forwards model. It includes the process of generating a surface, rotating it, and extracting astrometric and photometric signals. It also demonstrates the code, and notes where changes could be made to extend this project.

**Contents:**
1. Surface representation
2. Rotations via Wigner-D matrices
3. Measurement kernels (astrometric & photometric)
4. The three-matrix forward model
5. Spherical cap spot model

Appendix A gives the notation, Appendix B maps each tag to the function that implements it


## 1. Surface representation

A stellar surface intensity map is a real-valued function on the sphere:

$$S(\theta, \phi) = \sum_{\ell=0}^{L} \sum_{m=-\ell}^{\ell} s^m_\ell \, Y^m_\ell(\theta, \phi) \tag{S1}$$

where $\theta \in [0, \pi]$ is the polar angle (colatitude) measured from the spin axis $+\hat{z}$, $\phi \in [-\pi, \pi]$ is
the azimuthal angle measured from $+\hat{x}$, and $Y^m_\ell$ are the complex spherical harmonics, WITH the Conden-Shortley phase, which comes from `scipy.special.sph_harm_y`. Latitude is $90^\circ -\theta$ and north is $+\hat{z}$. The stellar inclination $i$ is the angle between the spin axis and the line of sight, so $i=0^\circ$ is pole on, and $i=90^\circ$ is equator-on. Later, and throughout the paper, we reparameterise the geometery with $\beta=90^\circ - i$, so that $\beta=0$ is equator on, which is more natura w.r.t. the rotations that will be performed.


The coefficients are obtained from the analysis equation:

$$s^m_\ell = \int_0^{2\pi} \int_0^{\pi} S(\theta, \phi) \, Y^{m*}_\ell(\theta, \phi) \, \sin\theta \, d\theta \, d\phi \tag{S2}$$

This works because spherical harmonics are orthonormal:
$\int Y^m_\ell \, Y^{m'*}_{\ell'} \, d\Omega = \delta_{\ell\ell'} \delta_{mm'}$. A uniform map of intensity $I$ has $s^0_0 = 2\sqrt\pi\, I$


### Coefficient vector

We store all coefficients in a single vector, ordered by degree then order:

$$\mathbf{s} = [\underbrace{s_0^0}_{\ell=0},\; \underbrace{s_1^{-1}, s_1^0, s_1^1}_{\ell=1},\; \underbrace{s_2^{-2}, \ldots, s_2^2}_{\ell=2},\; \ldots] \tag{S3}$$

This vector has $(L+1)^2$ entries. The flat index for $(\ell, m)$ is $\ell^2 + \ell + m$.

### Reality constraint

Since $S(\theta, \phi) \in \mathbb{R}$, we use complex spherical harmonics for rotations
but enforce:

$$s_\ell^{-m} = (-1)^m \, (s_\ell^m)^* \tag{S4}$$

so $s^0_\ell$ is real and the $m < 0$ half is determined by the $m > 0$ half. The stored
real vector keeps only the independent parts,

$$\mathbf{s}_{\rm real} = [\, s^0_\ell \;|\; \mathrm{Re}\, s^m_\ell \;|\; \mathrm{Im}\, s^m_\ell \,], \qquad m > 0 \tag{S5}$$

each block ordered by $\ell$ then $m$, again $(L+1)^2$ real numbers.

Implemented by `indexing.lm_to_idx`, `indexing.idx_to_lm`, `indexing.lm_indices`, `indexing.n_coeffs` (S3) and
`indexing.coeffs_to_real`, `indexing.real_to_coeffs` (S4, S5).